## Primitive Logic --> Stateless Logic Circuits

- We've already see one level of upward abstraction in the previous section
    - PMOS/NMOS are just primitive on/off switches, but they combine to give you logical expressions
    - Logical expressions can be combined to give you novel logical behaviour (NOR + NAND + AND gives you XOR)

- Now, let's see how these primitive logic gates can get us to stateless operations! We will go through the implementations for the core primitives:
    - Adders
    - Multiplexers
    - Decoders
    - Comparators
    - Barrel Shifters & Rotators

- These primitives will give us some useful conceptual extensions, which we will also look at:
    - Subtractors
    - Multipliers
    - Encoders & Priority Encoders

- Finally, we will see how the combinations of these will jointly make up the Arithmetic Logic Unit (ALU) of a computer

### Adders

- Adders are the basic building blocks of all computer arithmetic. By chaining primitive logic gates together, we can translate Boolean logic directly into binary addition!

- There are 3 types of adders we will study
    - Half Adder
        - A half adder performs a sum of 2 bits
        - It outputs a sum bit and a carry bit
        - We consider it a half adder because it only outputs a carry bit, but doesn't accept a carry bit
    - Full Adder
        - A full adder performs the sum of 2 bits, AND a carry bit
        - It outputs a sum bit and a carry bit
    - Ripple Adder
        - This is a composite adder, which connects $N$ Full Adders in series to add $N$-bit numbers        
        - The carry output from each bit position "ripples" into the carry input of the next higher bit position

        

In [ ]:
from utils import *

def half_adder(a: TRANSISTOR_OUTPUT, b: TRANSISTOR_OUTPUT) -> tuple[TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT]:
    '''
            A ───┬─────────┐
                 │  ┌───┐  ├─── [cmos_XOR] ─── Sum
            B ───┼──┤XOR│──┘
                 │  └───┘
                 │  ┌───┐
                 └──┤AND│────── [cmos_AND] ─── Carry
                    └───┘
    '''
    # Sum is 1 if inputs differ; Carry is 1 if both inputs are 1
    sum_out = cmos_XOR(a, b)
    carry_out = cmos_AND(a, b)
    return sum_out, carry_out


def full_adder(a: TRANSISTOR_OUTPUT, b: TRANSISTOR_OUTPUT, c_in: TRANSISTOR_OUTPUT = GROUND) -> tuple[TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT]:
    '''
        A, B ──────> [Half Adder 1] ─── (Sum1, Carry1)
                           │
        Sum1, C_in ─> [Half Adder 2] ─── (Final Sum, Carry2)
                           │
        Carry1, Carry2 ─> [cmos_OR] ─── Final Carry Out
    '''
    # Stage 1: Add inputs A and B
    sum1, carry1 = half_adder(a, b)
    
    # Stage 2: Add incoming carry to intermediate sum
    final_sum, carry2 = half_adder(sum1, c_in)
    
    # Final carry triggers if either half adder generated a carry
    final_carry = cmos_OR(carry1, carry2)
    return final_sum, final_carry


def ripple_carry_adder(a_bits: list[TRANSISTOR_OUTPUT], b_bits: list[TRANSISTOR_OUTPUT]) -> tuple[list[TRANSISTOR_OUTPUT], TRANSISTOR_OUTPUT]:
    '''
    Ripple-Carry Adder processing LSB to MSB.
    Takes two equal-length lists of bit signals (ordered LSB -> MSB).
    Returns (sum_bits, final_carry_out).
    '''
    carry: TRANSISTOR_OUTPUT = GROUND
    sum_bits: list[TRANSISTOR_OUTPUT] = []
    
    for bit_a, bit_b in zip(a_bits, b_bits):
        s, carry = full_adder(bit_a, bit_b, carry)
        sum_bits.append(s)
        
    return sum_bits, carry